In [1]:
%reset -f
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, gc, time, zipfile
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
# 将上一级目录添加到搜索路径中
import sys
sys.path.append(os.path.abspath(".."))
from evaluator import evaluate_all  # 确保导入了你上传的评估文件

from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, accuracy_score, brier_score_loss, recall_score, precision_score)
from scipy import stats
import numpy as np

# 预测模型
from sklearn.linear_model import LogisticRegression    # Logit模型
from sklearn.ensemble import RandomForestClassifier     # 随机森林
from xgboost import XGBClassifier                       # XGBoost

# 评估与预处理
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler        # Logit 必须进行标准化
from sklearn.metrics import accuracy_score
from evaluator import evaluate_all

In [7]:
import joblib
file_path = "/root/autodl-tmp/DATA/data_bundles.pkl"
data_bundles = joblib.load(file_path)

In [8]:
# 函数
def undersample_indices(y, neg_pos_ratio=1, seed=42):
    """对训练集进行随机欠采样（Random Undersampling）"""

    # 确保标签为整数型 0/1
    y = np.asarray(y).astype(int)

    # 分别获取正样本和负样本的索引位置
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    # 正样本数量
    n_pos = len(pos_idx)
    if n_pos == 0:
        raise ValueError("训练集中没有正样本，无法进行欠采样。")
    n_neg_need = min(len(neg_idx), n_pos * int(neg_pos_ratio))

    # 在负样本中随机无放回抽取所需数量
    rng = np.random.default_rng(seed)
    neg_sel = rng.choice(neg_idx, size=n_neg_need, replace=False)

    # 合并正样本与抽取的负样本索引
    sel = np.concatenate([pos_idx, neg_sel])
    # 打乱顺序，避免样本按类别排序
    rng.shuffle(sel)
    return sel

from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, accuracy_score, brier_score_loss)
from scipy import stats

# t 区间 CI（重复10次，df=9）
def t_ci(values, ci=0.95):
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    n = len(v)
    if n < 2:
        return (np.nan, np.nan, np.nan)
    mean = v.mean()
    se = v.std(ddof=1) / np.sqrt(n)
    alpha = 1 - ci
    tcrit = stats.t.ppf(1 - alpha/2, df=n-1)
    lo = mean - tcrit * se
    hi = mean + tcrit * se
    return mean, lo, hi

def fmt_ci(mean, lo, hi, nd=4):
    if np.isnan(mean):
        return "NA"
    return f"{mean:.{nd}f}[{lo:.{nd}f}; {hi:.{nd}f}]"


metric_names = [
    "ROC-AUC", "F1", "Recall@1%", "Recall@5%"
]

def fit_predict_model(model, X_train, y_train, X_test, threshold=0.5):
    """
    统一封装：训练 + 预测概率 + 生成预测标签, 只需要替换 model 的构造即可
    """
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(np.int8)

    # 某些模型（如logit）有 n_iter_，树模型没有
    n_iter = getattr(model, "n_iter_", None)
    if isinstance(n_iter, (list, np.ndarray)):
        n_iter = n_iter[0]
    return y_prob, y_pred, n_iter

In [11]:
from evaluator import evaluate_all
from joblib import Parallel, delayed

# =========================
# 参数
# =========================
target = "insider_trading"
sets = ["set1","set2","set3"]
neg_pos_ratios = [1, 2, 5]
n_rep = 10
ci_level = 0.95
base_seed = 42
PARALLEL_JOBS = 20  # 建议先从 4~8 试起，稳定后再上调

main_cols = ['insider_trading', 'Stkcd', 'Trddt']
features = [col for col in data_bundles['set1']['train'].columns if col not in main_cols]
# =========================
# 输出的指标列
# =========================
base_metrics = ["ROC-AUC", "F1"]
tail_metrics = ["Recall@0.10%","Recall@1.00%"]
metric_names = base_metrics + tail_metrics

# =========================
# 单次 run：欠采样 -> 训练 -> 预测 -> evaluate_all
# 注意：并行阶段 fig_path 必须传 None（不要在子进程画图）
# =========================
def one_run_rf(seed, ratio, X_train_full, y_train_full, X_test, y_test, use_balanced=False):
    # ① 欠采样（只作用训练集）
    sel = undersample_indices(y_train_full, neg_pos_ratio=ratio, seed=seed)
    X_train = X_train_full[sel]
    y_train = y_train_full[sel]

    # ② class_weight（欠采样时一般不建议 balanced）
    cw = "balanced" if use_balanced else None

    # 默认参数
    model = RandomForestClassifier(
        n_estimators=100,
        criterion="gini",
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        min_weight_fraction_leaf=0.0,
        max_features="sqrt",
        max_leaf_nodes=None,
        min_impurity_decrease=0.0,
        bootstrap=True,
        oob_score=False,
        n_jobs=1,
        random_state=seed,
        class_weight=None,
        max_samples=None
    )
    
    model.fit(X_train, y_train)

    # ④ 预测概率
    y_prob = model.predict_proba(X_test)[:, 1]

    # ⑤ 新评估器（run-level 原始指标）
    metrics, _ = evaluate_all(
        y_true=y_test,
        y_prob=y_prob,
        base_thr=0.5,
        best_metric="f1", 
        fig_path=None    
    )
    return metrics, None

# =========================
# 主过程：每个 set 输出一张表
# =========================
tables = {}

for set_name in sets:
    print(f"\n==================== {set_name} ====================")

    train_df = data_bundles[set_name]["train"]
    test_df  = data_bundles[set_name]["test"]

    # 数据准备：一次性转 numpy
    X_train_full = train_df[features].to_numpy(dtype=np.float32, copy=False)
    y_train_full = train_df[target].to_numpy(dtype=np.int8, copy=False)

    X_test = test_df[features].to_numpy(dtype=np.float32, copy=False)
    y_test = test_df[target].to_numpy(dtype=np.int8, copy=False)

    print("train shape:", X_train_full.shape, "pos_rate:", float(y_train_full.mean()))
    print("test  shape:", X_test.shape, "pos_rate:", float(y_test.mean()))

    rows = []

    for ratio in neg_pos_ratios:
        print(f"\n--- 欠采样 负:正={ratio}:1 | 并行重复 {n_rep} 次 ---")

        store = {k: [] for k in metric_names}
        seeds = [base_seed + r for r in range(n_rep)]

        results = Parallel(
            n_jobs=PARALLEL_JOBS,
            backend="loky",
            verbose=0
        )(
            delayed(one_run_rf)(
                seed, ratio,
                X_train_full, y_train_full,
                X_test, y_test
            )
            for seed in seeds
        )

      # 汇总每次 run 的指标
        for i, (met, n_iter) in enumerate(results, start=1):
            for k in metric_names:
                store[k].append(met.get(k, np.nan))

            # 提取指标用于打印（注意 Key 名要和 evaluator 内部一致）
            auc_val = met.get("ROC-AUC", np.nan)
            f1_val  = met.get("F1", np.nan)
            r1_val  = met.get("Recall@0.1%", np.nan) # 注意此处带 .00
            r5_val  = met.get("Recall@1%", np.nan)
            
            metrics_str = (f"AUC={auc_val:.4f} | F1={f1_val:.4f} | "
                           f"R@1%={r1_val:.4f} | R@5%={r5_val:.4f}")

            # 打印逻辑（现在 n_iter 会是 None）
            if n_iter is None:
                print(f"  run {i:02d}/{n_rep} | {metrics_str}")
            else:
                it = n_iter[0] if isinstance(n_iter, (list, np.ndarray)) else n_iter
                print(f"  run {i:02d}/{n_rep} | n_iter={it} | {metrics_str}")        

        # 只输出均值，不输出95%CI
        row = {"Train undersample (neg:pos)": f"{ratio}:1"}
        for k in metric_names:
            row[k] = round(np.nanmean(store[k]), 4)
        rows.append(row)

    table = pd.DataFrame(rows)
    tables[set_name] = table

    print(f"\n>>> {set_name} 汇总表：")
    display(table)


==================== set1 ====================
train shape: (3028146, 720) pos_rate: 0.0014163782063348332
test  shape: (1046498, 720) pos_rate: 0.00017391337584973885

--- 欠采样 负:正=1:1 | 并行重复 10 次 ---
  run 01/10 | AUC=0.5825 | F1=0.1529 | R@1%=0.4780 | R@5%=0.4780
  run 02/10 | AUC=0.6695 | F1=0.1529 | R@1%=0.4780 | R@5%=0.4780
  run 03/10 | AUC=0.6929 | F1=0.1472 | R@1%=0.4780 | R@5%=0.4835
  run 04/10 | AUC=0.7351 | F1=0.1378 | R@1%=0.4780 | R@5%=0.4780
  run 05/10 | AUC=0.7352 | F1=0.1505 | R@1%=0.4780 | R@5%=0.4780
  run 06/10 | AUC=0.8483 | F1=0.1529 | R@1%=0.4780 | R@5%=0.7857
  run 07/10 | AUC=0.8861 | F1=0.1529 | R@1%=0.4780 | R@5%=0.4835
  run 08/10 | AUC=0.7623 | F1=0.1529 | R@1%=0.4780 | R@5%=0.4780
  run 09/10 | AUC=0.7218 | F1=0.1529 | R@1%=0.4780 | R@5%=0.4780
  run 10/10 | AUC=0.7165 | F1=0.1529 | R@1%=0.4780 | R@5%=0.4780

--- 欠采样 负:正=2:1 | 并行重复 10 次 ---
  run 01/10 | AUC=0.6657 | F1=0.1529 | R@1%=0.4780 | R@5%=0.4780
  run 02/10 | AUC=0.7944 | F1=0.1529 | R@1%=0.4780

,Train undersample (neg:pos),ROC-AUC,F1,Recall@0.10%,Recall@1.00%
0,1:1,0.7350,0.1506,0.4780,0.478
1,2:1,0.7474,0.1476,0.4747,0.478
2,5:1,0.7321,0.1377,0.4632,0.478



==================== set2 ====================
train shape: (3028146, 720) pos_rate: 0.002427557984324402
test  shape: (1046498, 720) pos_rate: 0.00025131438378286436

--- 欠采样 负:正=1:1 | 并行重复 10 次 ---
  run 01/10 | AUC=0.6600 | F1=0.1155 | R@1%=0.4411 | R@5%=0.4411
  run 02/10 | AUC=0.6821 | F1=0.1206 | R@1%=0.4411 | R@5%=0.4411
  run 03/10 | AUC=0.5693 | F1=0.1155 | R@1%=0.4411 | R@5%=0.4411
  run 04/10 | AUC=0.7645 | F1=0.1189 | R@1%=0.4411 | R@5%=0.4411
  run 05/10 | AUC=0.7504 | F1=0.1190 | R@1%=0.4411 | R@5%=0.4411
  run 06/10 | AUC=0.7444 | F1=0.1174 | R@1%=0.4411 | R@5%=0.4411
  run 07/10 | AUC=0.8009 | F1=0.1155 | R@1%=0.4411 | R@5%=0.4411
  run 08/10 | AUC=0.5709 | F1=0.1155 | R@1%=0.4411 | R@5%=0.4411
  run 09/10 | AUC=0.8465 | F1=0.1155 | R@1%=0.4411 | R@5%=0.4639
  run 10/10 | AUC=0.6718 | F1=0.1155 | R@1%=0.4411 | R@5%=0.4411

--- 欠采样 负:正=2:1 | 并行重复 10 次 ---
  run 01/10 | AUC=0.6545 | F1=0.1155 | R@1%=0.4411 | R@5%=0.4411
  run 02/10 | AUC=0.8445 | F1=0.1188 | R@1%=0.4411 

,Train undersample (neg:pos),ROC-AUC,F1,Recall@0.10%,Recall@1.00%
0,1:1,0.7061,0.1169,0.3008,0.4411
1,2:1,0.7108,0.1175,0.2563,0.4411
2,5:1,0.6954,0.1196,0.2190,0.4411



==================== set3 ====================
train shape: (3028146, 720) pos_rate: 0.0016032912547809782
test  shape: (1046498, 720) pos_rate: 0.0008007659833081382

--- 欠采样 负:正=1:1 | 并行重复 10 次 ---
  run 01/10 | AUC=0.8309 | F1=0.2807 | R@1%=0.6074 | R@5%=0.6074
  run 02/10 | AUC=0.8054 | F1=0.2809 | R@1%=0.6074 | R@5%=0.6158
  run 03/10 | AUC=0.8580 | F1=0.2662 | R@1%=0.6074 | R@5%=0.6957
  run 04/10 | AUC=0.8660 | F1=0.2809 | R@1%=0.6074 | R@5%=0.6134
  run 05/10 | AUC=0.8198 | F1=0.2664 | R@1%=0.6074 | R@5%=0.6074
  run 06/10 | AUC=0.8384 | F1=0.2664 | R@1%=0.5847 | R@5%=0.6074
  run 07/10 | AUC=0.8387 | F1=0.2724 | R@1%=0.6074 | R@5%=0.6074
  run 08/10 | AUC=0.7995 | F1=0.2662 | R@1%=0.6074 | R@5%=0.6086
  run 09/10 | AUC=0.8226 | F1=0.2809 | R@1%=0.5346 | R@5%=0.6086
  run 10/10 | AUC=0.7937 | F1=0.2662 | R@1%=0.6074 | R@5%=0.6074

--- 欠采样 负:正=2:1 | 并行重复 10 次 ---
  run 01/10 | AUC=0.8214 | F1=0.2809 | R@1%=0.5394 | R@5%=0.6158
  run 02/10 | AUC=0.7853 | F1=0.2822 | R@1%=0.6074 

,Train undersample (neg:pos),ROC-AUC,F1,Recall@0.10%,Recall@1.00%
0,1:1,0.8273,0.2727,0.3791,0.5979
1,2:1,0.7830,0.2782,0.3770,0.5856
2,5:1,0.7925,0.2795,0.3729,0.5563


In [12]:
SAVE_EXCEL = True
excel_path = "/root/autodl-tmp/Table5/rf_tables_without_tuning.xlsx"

# 自动创建路径中不存在的文件夹，防止路径错误
import os
os.makedirs(os.path.dirname(excel_path), exist_ok=True)

# 保存到Excel（每个set一个sheet）
if SAVE_EXCEL:
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        for set_name, df_out in tables.items():
            df_out.to_excel(writer, sheet_name=set_name, index=False)
    print("\n已保存：", excel_path)


已保存： /root/autodl-tmp/Table5/rf_tables_without_tuning.xlsx
